# SFT Stage 2 混合身份注入数据准备

## 功能
1. 从 stage1 通用数据中按 9:1 比例随机抽取数据
2. 统一身份数据格式（messages → conversations）
3. 混合 stage1 抽取数据 + stage2 身份数据
4. **抽取**（非复制）：被抽走的数据将从 stage1 原文件中移除

## 数据流
```
stage1 (~190万条)  ──→  抽取 9×N   ──→  混合 (9+1)×N  ──→  sft_stage2_mixed/
                   ──→  剩余数据    ──→  sft_t2t_cleaned_v8_remaining.jsonl
stage2 (~1.8万条)  ──→  格式统一   ──┘
```

In [ ]:
# ============================================================
# Cell 1: 配置参数
# ============================================================
import json, random, os
from pathlib import Path
from collections import Counter

# ---- 云端路径配置 ----
STAGE1_FILE = Path("/mnt/workspace/shayler2.0/data_stage1/sft_t2t_cleaned_v8_fixed.jsonl")
STAGE2_DIR  = Path("/mnt/workspace/shayler2.0/data_stage2")
OUTPUT_DIR  = Path("/mnt/workspace/shayler2.0/data_stage2_mixed")

# ---- 混合比例 ----
RATIO = 9          # stage1 : stage2 = 9 : 1
SEED  = 42         # 随机种子，确保可复现

random.seed(SEED)

print(f"Stage1 文件: {STAGE1_FILE}")
print(f"Stage2 目录: {STAGE2_DIR}")
print(f"输出目录:    {OUTPUT_DIR}")
print(f"混合比例:    {RATIO}:1")
print(f"随机种子:    {SEED}")

In [ ]:
# ============================================================
# Cell 2: 加载 Stage2 身份数据 + 格式统一
# ============================================================

def convert_to_conversations(entry: dict) -> dict:
    """将 messages 格式统一为 conversations 格式"""
    if "conversations" in entry:
        return entry
    if "messages" in entry:
        msgs = entry["messages"]
        convs = []
        for m in msgs:
            role = m.get("role", "")
            # 角色统一：user 保持不变，assistant/小乐 统一为 assistant
            new_role = "user" if role == "user" else "assistant"
            convs.append({"role": new_role, "content": m.get("content", "")})
        return {"conversations": convs}
    # 跳过无法识别的格式
    return None

# 读取所有身份数据文件（sft_*.jsonl，排除 dpo_pairs.jsonl）
identity_files = sorted(STAGE2_DIR.glob("sft_*.jsonl"))
print(f"找到 {len(identity_files)} 个身份数据文件:")
for fp in identity_files:
    print(f"  {fp.name}")

all_identity = []
file_counts = {}
for fp in identity_files:
    count = 0
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            entry = convert_to_conversations(entry)
            if entry is None:
                continue
            all_identity.append(entry)
            count += 1
    file_counts[fp.name] = count
    print(f"  {fp.name}: {count} 条")

N_IDENTITY = len(all_identity)
print(f"\n身份数据总计: {N_IDENTITY} 条")
print(f"格式: 全部统一为 conversations (user/assistant)")

In [ ]:
# ============================================================
# Cell 3: 计算抽取量
# ============================================================

NEED_STAGE1 = N_IDENTITY * RATIO
TOTAL_MIXED = NEED_STAGE1 + N_IDENTITY

print(f"身份数据 (stage2):  {N_IDENTITY:>8} 条  (比例 1)")
print(f"需从 stage1 抽取:   {NEED_STAGE1:>8} 条  (比例 {RATIO})")
print(f"混合后总量:         {TOTAL_MIXED:>8} 条")
print(f"实际比例:           {NEED_STAGE1/N_IDENTITY:.2f}:1")

In [ ]:
# ============================================================
# Cell 4: 统计 stage1 行数 + 随机选取行号
# ============================================================

print("统计 stage1 总行数...")
total_lines = 0
with open(STAGE1_FILE, "r", encoding="utf-8") as f:
    for _ in f:
        total_lines += 1

print(f"stage1 总行数: {total_lines:,}")

if NEED_STAGE1 > total_lines:
    raise ValueError(f"抽取量 ({NEED_STAGE1}) 超过总量 ({total_lines})！")

# 随机选取行号（set + while 循环，适用于所有 Python 版本）
print(f"随机选取 {NEED_STAGE1:,} 个行号...")
selected = set()
while len(selected) < NEED_STAGE1:
    # 批量生成减少循环开销
    batch_size = min(NEED_STAGE1 - len(selected), 10000)
    for _ in range(batch_size):
        selected.add(random.randint(0, total_lines - 1))
    if len(selected) % 50000 == 0:
        pct = len(selected) / NEED_STAGE1 * 100
        print(f"  进度: {len(selected):,} / {NEED_STAGE1:,} ({pct:.0f}%)")

print(f"已选取 {len(selected):,} 个行号")

In [ ]:
# ============================================================
# Cell 5: 分割 stage1 — 抽取部分 vs 剩余部分
# ============================================================
# 被选中的行 → 抽取文件（后续混入 stage2）
# 未被选中的行 → 剩余文件（替换原 stage1 文件）

STAGE1_EXTRACTED = STAGE1_FILE.parent / f"{STAGE1_FILE.stem}_extracted.jsonl"
STAGE1_REMAINING = STAGE1_FILE.parent / f"{STAGE1_FILE.stem}_remaining.jsonl"

print(f"抽取文件: {STAGE1_EXTRACTED}")
print(f"剩余文件: {STAGE1_REMAINING}")
print()

extracted_entries = []
n_extracted = 0
n_remaining = 0

print("开始分割...")
with open(STAGE1_FILE, "r", encoding="utf-8") as fin, \
     open(STAGE1_EXTRACTED, "w", encoding="utf-8") as f_ext, \
     open(STAGE1_REMAINING, "w", encoding="utf-8") as f_rem:
    
    for i, line in enumerate(fin):
        if i in selected:
            f_ext.write(line)
            try:
                extracted_entries.append(json.loads(line))
            except json.JSONDecodeError:
                pass
            n_extracted += 1
        else:
            f_rem.write(line)
            n_remaining += 1
        
        if (i + 1) % 300000 == 0:
            pct = (i + 1) / total_lines * 100
            print(f"  已处理 {i+1:,}/{total_lines:,} ({pct:.0f}%)  |  抽取 {n_extracted:,}  |  剩余 {n_remaining:,}")

print(f"\n完成!")
print(f"  抽取: {n_extracted:,} 条 → {STAGE1_EXTRACTED.name}")
print(f"  剩余: {n_remaining:,} 条 → {STAGE1_REMAINING.name}")
print(f"  验证: {n_extracted + n_remaining:,} = {total_lines:,} {'✓' if n_extracted + n_remaining == total_lines else '✗ 不一致!'}")

In [ ]:
# ============================================================
# Cell 6: 混合 + 打乱 + 输出
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 合并
all_mixed = all_identity + extracted_entries
print(f"合并: identity={len(all_identity):,} + stage1_extracted={len(extracted_entries):,} = {len(all_mixed):,}")

# 打乱
random.shuffle(all_mixed)
print("打乱完成")

# 输出
OUTPUT_FILE = OUTPUT_DIR / "sft_stage2_mixed.jsonl"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for entry in all_mixed:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"混合数据已写入: {OUTPUT_FILE}")
print(f"文件大小: {OUTPUT_FILE.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# ============================================================
# Cell 7: 统计报告 + 验证
# ============================================================

print("=" * 60)
print("  SFT Stage 2 数据混合报告")
print("=" * 60)
print(f"  Stage 1 原始总量:  {total_lines:>10,}")
print(f"  Stage 1 抽取量:    {NEED_STAGE1:>10,}  (比例 {RATIO})")
print(f"  Stage 1 剩余量:    {total_lines - NEED_STAGE1:>10,}")
print(f"  Stage 2 身份量:    {N_IDENTITY:>10,}  (比例 1)")
print(f"  ─────────────────────────────")
print(f"  混合后总量:        {len(all_mixed):>10,}")
print(f"  实际比例:          {NEED_STAGE1/N_IDENTITY:.2f}:1")
print()
print(f"  输出文件: {OUTPUT_FILE}")
print(f"  Stage1 剩余: {STAGE1_REMAINING}")
print(f"  Stage1 抽取: {STAGE1_EXTRACTED}")

# 格式验证
print()
print("-" * 60)
print("  格式验证")
print("-" * 60)
fmt_counter = Counter()
role_counter = Counter()
for entry in all_mixed:
    if "conversations" in entry:
        fmt_counter["conversations"] += 1
        for m in entry["conversations"]:
            role_counter[m.get("role", "?")] += 1
    elif "messages" in entry:
        fmt_counter["messages"] += 1

print(f"  数据格式: {dict(fmt_counter)}")
print(f"  角色分布: {dict(role_counter)}")
if "messages" in fmt_counter:
    print("  ⚠ 警告：仍有 messages 格式数据未转换！")
else:
    print("  ✓ 格式统一为 conversations")

# 随机抽样展示
print()
print("-" * 60)
print("  随机抽样检查 (3 条)")
print("-" * 60)
samples = random.sample(all_mixed, min(3, len(all_mixed)))
for i, entry in enumerate(samples):
    msgs = entry.get("conversations", entry.get("messages", []))
    roles = [m.get("role", "?") for m in msgs[:3]]
    preview = msgs[0].get("content", "")[:50] if msgs else "(空)"
    print(f"  [{i+1}] roles={roles} | turns={len(msgs)} | preview={preview}...")

print()
print("=" * 60)
print("  数据准备完成 ✓")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 8: 替换原 stage1 文件（确认后执行）
# ============================================================
# ⚠ 此操作会用「剩余数据」替换「原 stage1 文件」
# 被抽取的 163,620 条数据将不再存放于 stage1 文件中
# 执行前请确认 Cell 5 和 Cell 7 验证通过

import shutil

# 1. 备份原文件
BACKUP_FILE = STAGE1_FILE.with_suffix(".jsonl.bak_before_mix")
print(f"备份原文件 → {BACKUP_FILE}")
shutil.move(str(STAGE1_FILE), str(BACKUP_FILE))

# 2. 剩余数据替换为原文件
print(f"剩余数据 → {STAGE1_FILE}")
shutil.move(str(STAGE1_REMAINING), str(STAGE1_FILE))

# 3. 清理抽取临时文件（数据已在混合文件中）
STAGE1_EXTRACTED.unlink()
print(f"清理抽取临时文件: {STAGE1_EXTRACTED.name}")

print()
print("替换完成！原 stage1 文件已备份为 .bak_before_mix")
print(f"stage1 文件现包含 {n_remaining:,} 条数据（原 {total_lines:,} - 抽取 {NEED_STAGE1:,}）")

## 执行步骤

1. **依次执行 Cell 1-7**（数据准备 + 验证，不修改原文件）
2. **检查 Cell 7 的验证报告**，确认：
   - 格式统一为 `conversations`（无残留 `messages`）
   - 总条数 = 身份数据 + 抽取数据
   - 分割验证 ✓
3. **确认无误后，执行 Cell 8**（替换原 stage1 文件）

## 输出文件

| 文件 | 说明 |
|------|------|
| `data_stage2_mixed/sft_stage2_mixed.jsonl` | 混合后的 Stage 2 训练数据 |
| `data_stage1/sft_t2t_cleaned_v8_fixed_remaining.jsonl` | Stage 1 剩余数据（替换原文件后消失）|
| `data_stage1/sft_t2t_cleaned_v8_fixed.jsonl.bak_before_mix` | Stage 1 原文件备份 |